# DATA DOWNLOADING

In [ ]:
# --- Fast/robust fetch of Stanford Dogs into /content/stanford-dogs (Colab) ---
# --- Fast/robust fetch of Stanford Dogs into /content/stanford-dogs (Colab) ---

!sudo apt-get -y -qq install aria2 > /dev/null
%pip install sympy==1.12

import os, tarfile, subprocess, shutil, pathlib

root = "/content/stanford-dogs"
os.makedirs(root, exist_ok=True)
os.chdir(root)

# Canonical URLs (case-sensitive!)
urls = [
  "http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar",
  "http://vision.stanford.edu/aditya86/ImageNetDogs/annotation.tar",  # <-- Capital A
  "http://vision.stanford.edu/aditya86/ImageNetDogs/lists.tar",
]

def have_all_archives():
    return all(os.path.exists(p) for p in ["images.tar", "Annotation.tar", "lists.tar"])

def download_with_aria2(urls):
    print("Downloading with aria2c…")
    cmd = ["aria2c", "-x", "8", "-s", "8", "-c"] + urls
    return subprocess.run(cmd).returncode == 0

def download_with_wget(url):
    print(f"wget {url}")
    return subprocess.run(["wget", "-c", url]).returncode == 0

# Try aria2 once; if any file missing, retry missing ones with wget (sequential)
if not have_all_archives():
    download_with_aria2(urls)

for url in urls:
    fname = url.rsplit("/",1)[-1]
    if not os.path.exists(fname):
        download_with_wget(url)

# Final check
missing = [f for f in ["images.tar","annotation.tar","lists.tar"] if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"Missing archives after download: {missing}")

print("\nExtracting…")
for t in ["images.tar","annotation.tar","lists.tar"]:
    print("Extracting", t)
    with tarfile.open(t) as tf:
        tf.extractall(path=".")

# Sanity checks
imgs_ok = os.path.isdir(os.path.join(root,"Images"))
ann_ok  = os.path.isdir(os.path.join(root,"Annotation"))
# lists_ok= os.path.isdir(os.path.join(root,"lists"))

print("\nTree:")
print(" - Images exists?", imgs_ok)
print(" - Annotation exists?", ann_ok)
# print(" - lists exists?", lists_ok)

# Extra: verify the .mat files are present
mat_ok = (os.path.exists(os.path.join(root, "train_list.mat")) and
          os.path.exists(os.path.join(root, "test_list.mat")))
print(" - train_list.mat/test_list.mat present?", mat_ok)

print("\nSet CFG['data_root'] =", root)

# TEACHER MOBILENETV2 TRAINING --> TRAINING

In [ ]:
# ============================================================
# Colab: Stanford Dogs → MobileNetV2 backbone offline training
# ============================================================

# (Optional) Mount Drive if your data is there
# from google.colab import drive; drive.mount('/content/drive')

!pip -q install scipy pyyaml

import os, json, math, random, time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from scipy.io import loadmat

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torchvision import transforms, models

# -------------------------
# Config (edit as you like)
# -------------------------
CFG = {
    # <<< set this to your dataset location >>>
    "data_root": "/content/stanford-dogs/",   # e.g., "/content/drive/MyDrive/stanford-dogs"
    # outputs
    "cleaned_data_path_csv": "/content/dog_breeds_index.csv",
    "cleaned_data_path_json": "/content/dog_breeds_index.json",
    "artifacts_dir": "/content/backbone_artifacts",

    # backbone selection
    "random_seed": 42,
    "num_pretrain_dogs": 10,          # number of backbone classes to pretrain on

    # image + transforms
    "img_size": 224,

    # training
    "epochs_head_warmup": 3,          # head-only
    "epochs_finetune": 13,            # full fine-tune max
    "patience": 7,                    # early stopping
    "batch_size_train": 64,
    "batch_size_val": 64,
    "lr_head": 1e-3,
    "lr_finetune": 3e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.05,
}

# -------------------------
# Reproducibility helpers
# -------------------------
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CFG["random_seed"])

# -------------------------
# 1) Clean / index dataset
# -------------------------
def _parse_list(mat_path: Path):
    m = loadmat(mat_path)
    files  = [str(x[0]).strip() for x in m['file_list'].squeeze()]
    labels = [int(x) for x in m['labels'].squeeze()]  # 1..120
    return files, labels

def clean_dogs(data_root: str, out_csv: str, out_json: str):
    root = Path(data_root)
    if not (root / "train_list.mat").exists():
        raise FileNotFoundError(
            f"Could not find {root/'train_list.mat'}. "
            "Point CFG['data_root'] to your Stanford Dogs folder."
        )
    tr_files, tr_labels = _parse_list(root/"train_list.mat")
    te_files, te_labels = _parse_list(root/"test_list.mat")

    def rows(files, labels, split):
        for fp, y in zip(files, labels):
            breed = fp.split('/')[0]  # e.g., n02085620-Chihuahua
            yield dict(
                split=split,
                rel_path=fp,
                img_path=str(root/"Images"/fp),
                ann_path=str(root/"Annotation"/(fp.replace('.jpg',''))), # folder + xml name
                breed=breed,
                gid=int(y)-1   # 0..119
            )

    df = pd.DataFrame([*rows(tr_files,tr_labels,"train"), *rows(te_files,te_labels,"test")])
    df.to_csv(out_csv, index=False)

    gid2breed = df.groupby("gid")["breed"].first().sort_index().to_dict()
    Path(out_json).parent.mkdir(parents=True, exist_ok=True)
    Path(out_json).write_text(json.dumps(gid2breed, indent=2))
    print(f"[clean] wrote:\n  {out_csv}\n  {out_json}")
    return df, gid2breed

df_idx, gid2breed = clean_dogs(CFG["data_root"], CFG["cleaned_data_path_csv"], CFG["cleaned_data_path_json"])

# --------------------------------------
# 2) Pick backbone classes (by seed)
# --------------------------------------
def pick_backbone_gids(gid2breed: dict, num_pretrain: int, seed: int):
    all_gids = list(map(int, gid2breed.keys()))
    if num_pretrain > len(all_gids):
        raise ValueError(f"num_pretrain_dogs={num_pretrain} > total classes={len(all_gids)}")
    rng = random.Random(seed)
    all_shuf = all_gids[:]
    rng.shuffle(all_shuf)
    chosen = sorted(all_shuf[:num_pretrain])
    print("Backbone dogs:")
    for g in chosen:
        print(f"  gid={g:3d}  breed={gid2breed[g]}")
    return chosen

backbone_gids = pick_backbone_gids(gid2breed, CFG["num_pretrain_dogs"], CFG["random_seed"])

# --------------------------------------
# 3) Datasets / DataLoaders
# --------------------------------------
class DogCsvDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tfm, gid_to_local: dict[int,int]):
        self.df = df.reset_index(drop=True)
        self.tfm = tfm
        self.g2l = gid_to_local

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img = Image.open(r.img_path).convert("RGB")
        x = self.tfm(img)
        y_local = self.g2l[int(r.gid)]
        return x, y_local

def make_loaders(df_idx: pd.DataFrame, backbone_gids: list[int], img_size: int, bs_tr: int, bs_val: int, device: torch.device):
    # filter to backbone classes
    bb_df = df_idx[df_idx["gid"].isin(backbone_gids)].copy()

    # stable (0..B-1) label map
    local_ids = sorted(backbone_gids)
    gid_to_local = {g:i for i,g in enumerate(local_ids)}

    df_train = bb_df[bb_df["split"]=="train"].copy()
    df_val   = bb_df[bb_df["split"]=="test"].copy()

    mean, std = [0.485,0.456,0.406], [0.229,0.224,0.225]
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    ds_tr  = DogCsvDataset(df_train, train_tf, gid_to_local)
    ds_val = DogCsvDataset(df_val,   val_tf,   gid_to_local)

    pin = (device.type == "cuda")
    dl_tr  = DataLoader(ds_tr,  batch_size=bs_tr,  shuffle=True,  num_workers=2, pin_memory=pin)
    dl_val = DataLoader(ds_val, batch_size=bs_val, shuffle=False, num_workers=2, pin_memory=pin)

    return dl_tr, dl_val, gid_to_local, local_ids

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
dl_tr, dl_val, gid_to_local, local_ids = make_loaders(
    df_idx, backbone_gids, CFG["img_size"],
    CFG["batch_size_train"], CFG["batch_size_val"], device
)

# --------------------------------------
# 4) MobileNetV2 model (ImageNet init)
# --------------------------------------
def make_mobilenet_v2(num_classes: int, pretrained: bool = True):
    try:
        # torchvision >= 0.13 style
        weights = models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
        net = models.mobilenet_v2(weights=weights)
    except Exception:
        net = models.mobilenet_v2(pretrained=pretrained)
    # replace classifier head
    in_f = net.classifier[-1].in_features  # 1280
    net.classifier[-1] = nn.Linear(in_f, num_classes)
    return net, in_f

num_classes = len(local_ids)
model, feat_dim = make_mobilenet_v2(num_classes=num_classes, pretrained=True)
model.to(device)

# --------------------------------------
# 5) Train loop (head warmup → finetune)
# --------------------------------------
def per_class_accuracy(model, loader, local_ids, gid2breed, device):
    model.eval()
    per_tot = [0]*len(local_ids)
    per_cor = [0]*len(local_ids)
    tot = 0; cor = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            pred = logits.argmax(1)
            tot += yb.size(0)
            cor += (pred == yb).sum().item()
            for c in range(len(local_ids)):
                m = (yb == c)
                n = int(m.sum().item())
                if n > 0:
                    per_tot[c] += n
                    per_cor[c] += int((pred[m] == yb[m]).sum().item())
    overall = 100.0 * cor / max(1, tot)
    for i, gid in enumerate(local_ids):
        name = gid2breed[gid]
        acc = (100.0 * per_cor[i] / per_tot[i]) if per_tot[i] > 0 else 0.0
        print(f"  {name:35s}: {acc:5.1f}% ({per_cor[i]}/{per_tot[i]})")
    print(f"Overall val acc: {overall:.2f}%")
    return overall

def train_backbone_mnet(model, dl_tr, dl_val, device, cfg):
    label_smoothing = cfg["label_smoothing"]
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    # --- Phase 1: head-only warmup ---
    for p in model.features.parameters():
        p.requires_grad = False
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr_head"], weight_decay=cfg["weight_decay"])

    print("\n[Phase 1] Head-only warmup")
    model.train()
    for ep in range(cfg["epochs_head_warmup"]):
        tot, cor, loss_sum = 0, 0, 0.0
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
            tot += yb.size(0)
            cor += (logits.argmax(1) == yb).sum().item()
            loss_sum += float(loss.item())
        print(f"  epoch {ep:02d} | train_acc={100*cor/max(1,tot):.1f} | loss={loss_sum/len(dl_tr):.3f}")

    # --- Phase 2: full fine-tune with early stopping ---
    for p in model.features.parameters():
        p.requires_grad = True
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr_finetune"], weight_decay=cfg["weight_decay"])

    best_state = None
    best_val = 0.0
    patience = cfg["patience"]
    stall = 0

    print("\n[Phase 2] Full fine-tune")
    for ep in range(cfg["epochs_finetune"]):
        model.train()
        tot, cor, loss_sum = 0, 0, 0.0
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
            tot += yb.size(0)
            cor += (logits.argmax(1) == yb).sum().item()
            loss_sum += float(loss.item())
        tr_acc = 100*cor/max(1,tot)
        if ep % 1 == 0:
            print(f"[ep {ep:02d}] train_acc={tr_acc:.1f} | loss={loss_sum/len(dl_tr):.3f}")

        # validation
        model.eval()
        with torch.no_grad():
            v_tot, v_cor = 0, 0
            for xb, yb in dl_val:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                v_tot += yb.size(0)
                v_cor += (logits.argmax(1) == yb).sum().item()
            v_acc = 100*v_cor/max(1,v_tot)
        print(f"[ep {ep:02d}] val_acc={v_acc:.1f}")

        if v_acc > best_val:
            best_val = v_acc
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
            stall = 0

        else:
            stall += 1

        if stall >= patience:
            print(f"Early stopping at epoch {ep}: best val {best_val:.2f}%")
            break

    if best_state is not None:
        model.load_state_dict(best_state, strict=True)
    return model, best_val

print("\n============================")
print("Training MobileNetV2 backbone")
print("============================")
model, best_val = train_backbone_mnet(model, dl_tr, dl_val, device, CFG)

print("\n--- Per-class validation accuracy (backbone classes) ---")
_ = per_class_accuracy(model, dl_val, local_ids, gid2breed, device)

# --------------------------------------
# 6) Save artifacts for later use
# --------------------------------------
ART = Path(CFG["artifacts_dir"]); ART.mkdir(parents=True, exist_ok=True)
weights_path = ART/"mobilenetv2_backbone.pth"
meta_path    = ART/"backbone_meta.json"

torch.save(model.state_dict(), weights_path)
meta = {
    "backbone": "mobilenet_v2",
    "feature_dim": int(model.classifier[-1].in_features),  # 1280
    "num_backbone_classes": len(local_ids),
    "backbone_gids": local_ids,            # global class IDs used during backbone training
    "img_size": CFG["img_size"],
    "normalization": {"mean":[0.485,0.456,0.406], "std":[0.229,0.224,0.225]},
    "best_val_acc": float(best_val),
}
meta_path.write_text(json.dumps(meta, indent=2))

print(f"\nSaved:")
print(f"  • weights: {weights_path}")
print(f"  • meta   : {meta_path}")
print("\nDone!")


# TEACHER MOBILENETV2 TRAINING --> PROJECTION

In [ ]:
import torch, json
from pathlib import Path
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ART = Path("/content/backbone_artifacts")
weights_path = ART/"mobilenetv2_backbone.pth"
meta_path    = ART/"backbone_meta.json"

meta = json.loads(meta_path.read_text())
local_ids = meta["backbone_gids"]
img_size  = meta["img_size"]

# Rebuild same MobileNetV2 head size
teacher = models.mobilenet_v2(weights=None)
in_f = teacher.classifier[-1].in_features
teacher.classifier[-1] = torch.nn.Linear(in_f, len(local_ids))

# Load weights THEN move to device
state = torch.load(weights_path, map_location="cpu")
teacher.load_state_dict(state, strict=True)
teacher.to(device)           # <<< IMPORTANT
teacher.eval()

# ============================================================
# 7) CACHE teacher features+logits (deploy/weak transforms)
# ============================================================
print("\nCaching teacher features+logits with deploy transforms...")

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

mean, std = [0.485,0.456,0.406], [0.229,0.224,0.225]
deploy_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

class DogCsvDatasetWithKey(Dataset):
    def __init__(self, df, tfm, gid_to_local):
        self.df = df.reset_index(drop=True)
        self.tfm = tfm
        self.g2l = gid_to_local
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = self.tfm(Image.open(r.img_path).convert("RGB"))
        y = self.g2l[int(r.gid)]
        return x, y, r.img_path

bb_df = df_idx[df_idx["gid"].isin(local_ids)].copy()
gid_to_local = {g:i for i,g in enumerate(local_ids)}
df_tr = bb_df[bb_df["split"]=="train"].copy()
df_va = bb_df[bb_df["split"]=="test"].copy()

pin = (device.type == "cuda")
dl_tr_deploy = DataLoader(DogCsvDatasetWithKey(df_tr, deploy_tf, gid_to_local),
                          batch_size=CFG["batch_size_train"], shuffle=False,
                          num_workers=2, pin_memory=pin)
dl_va_deploy = DataLoader(DogCsvDatasetWithKey(df_va, deploy_tf, gid_to_local),
                          batch_size=CFG["batch_size_val"], shuffle=False,
                          num_workers=2, pin_memory=pin)

# single hook + buffer
_buf = {"z": None}
def _hook(module, inp, out):
    _buf["z"] = inp[0].detach()

h = teacher.classifier[-1].register_forward_hook(_hook)

@torch.no_grad()
def cache_pass(loader):
    feats, logits, ys, keys = [], [], [], []
    for xb, yb, kb in loader:
        xb = xb.to(device, non_blocking=True)   # inputs on same device as teacher
        out = teacher(xb)                        # forward triggers hook
        z = _buf["z"]                            # [N, 1280] on device
        feats.append(z.cpu())
        logits.append(out.cpu())
        ys.append(yb.clone())
        keys.extend(kb)
    return {
        "keys": keys,
        "y_local": torch.cat(ys, 0),
        "features": torch.cat(feats, 0),
        "logits": torch.cat(logits, 0),
        "meta": {"backbone_gids": local_ids, "img_size": img_size,
                 "normalization": {"mean": mean, "std": std}}
    }

train_cache = cache_pass(dl_tr_deploy)
val_cache   = cache_pass(dl_va_deploy)
h.remove()

cache_dir = ART / "teacher_cache"
cache_dir.mkdir(parents=True, exist_ok=True)
torch.save(train_cache, cache_dir / "train_cache.pt")
torch.save(val_cache,   cache_dir / "val_cache.pt")
print("Saved teacher caches:")
print(f"  • {cache_dir/'train_cache.pt'}  (features {tuple(train_cache['features'].shape)}, logits {tuple(train_cache['logits'].shape)})")
print(f"  • {cache_dir/'val_cache.pt'}    (features {tuple(val_cache['features'].shape)}, logits {tuple(val_cache['logits'].shape)})")

# DISTILLATION PART 2

In [ ]:
# distillation_1.py
# (cache KD) linear projector
'''
 learn a projector that maps teacher penultimate space (1280-D) → student feature space
 can use both teacher logits and teacher features

 Outputs:
 - proj_from_teacher_penult: weights of the learned projector (Linear 1280→128).
    - This tells the student what “good” 128-D features should look like (in teacher’s sense)!!
 -kd_head: a small linear head (128→B) trained in feature space.
	-Useful for monitoring / sanity ( on CIL I will use my own TDM head)
 '''



import json
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

### distillation helpers (move to utils later)


class StudentOnCache(nn.Module):
    def __init__(self, in_dim, feat_dim, num_classes, drop):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim, elementwise_affine=False)
        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(in_dim, feat_dim))
        self.kd_head = nn.Linear(feat_dim, num_classes)
    def forward(self, z_teacher_penult):
        z_in = self.norm(z_teacher_penult)
        z = self.proj(z_in)
        logits = self.kd_head(z)
        return logits, z


def kd_loss(student_logits, teacher_logits, T=2.0):
    return (T*T) * F.kl_div(
        F.log_softmax(student_logits/T, dim=1),
        F.softmax(teacher_logits/T, dim=1),
        reduction="batchmean",
    )
@torch.no_grad()
def log_logit_std(model, loader, device, prefix="train", max_batches=2):
    """Logs avg per-sample std of logits for student vs teacher."""
    was_training = model.training
    model.eval()                       # disable dropout for clean stats

    s_std_list, t_std_list = [], []
    batches = 0
    for xb, _, tb in loader:
        xb = xb.to(device)
        tb = tb.to(device)

        slogits, _ = model(xb)         # [N, C]
        # per-sample std across classes
        s_std = slogits.std(dim=1)     # [N]
        t_std = tb.std(dim=1)          # [N]

        s_std_list.append(s_std.mean().item())
        t_std_list.append(t_std.mean().item())

        batches += 1
        if batches >= max_batches:
            break

    if was_training:
        model.train()

    s_mean = sum(s_std_list) / max(1, len(s_std_list))
    t_mean = sum(t_std_list) / max(1, len(t_std_list))
    ratio = s_mean / max(1e-8, t_mean)
    print(f"[{prefix}] avg per-sample logit std: student≈{s_mean:.3f}, teacher≈{t_mean:.3f} (ratio≈{ratio:.2f})")
# -------------- per-class report --------------
@torch.no_grad()
def eval_per_class(model, loader, num_classes, device):
    tot = [0]*num_classes
    cor = [0]*num_classes
    model.eval()
    for xb, yb, _ in loader:
        xb = xb.to(device); yb = yb.to(device)
        slogits, _ = model(xb)
        pred = slogits.argmax(1)
        for c in range(num_classes):
            m = (yb == c)
            n = int(m.sum().item())
            if n > 0:
                tot[c] += n
                cor[c] += int((pred[m] == yb[m]).sum().item())
    return [100.0*cor[c]/tot[c] if tot[c]>0 else 0.0 for c in range(num_classes)]


# -------------- train --------------
def train(student, opt, distillation_epochs, device, tr_dl, va_dl, alpha_kd, alpha_ce, kd_T, ce, alpha_feat, mse, distillation_patience, num_classes):
    best_val, best_state, stall = 0.0, None, 0
    for ep in range(distillation_epochs):
        student.train()
        loss_sum = 0.0
        for batch_idx, (xb, yb, tb) in enumerate(tr_dl):
            xb = xb.to(device); yb = yb.to(device); tb = tb.to(device)
            slogits, z = student(xb)
            if ep==0 and batch_idx==0:
                with torch.no_grad():
                    s_std = slogits.std(dim=1).mean().item()
                    t_std = tb.std(dim=1).mean().item()
                    print(f"[sanity] avg per-sample logit std: student≈{s_std:.3f}, teacher≈{t_std:.3f}")
            loss = alpha_kd*kd_loss(slogits, tb, kd_T) \
                + alpha_ce*ce(slogits, yb)
            # use cosine instead of mse
            if alpha_feat > 0:
                loss += alpha_feat * (1.0 - F.cosine_similarity(
                    F.normalize(z, dim=1), F.normalize(xb, dim=1), dim=1
                ).mean())
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            loss_sum += float(loss.item())

        # val
        student.eval()
        tot = cor = 0
        with torch.no_grad():
            for xb, yb, _ in va_dl:
                xb = xb.to(device); yb = yb.to(device)
                slogits, _ = student(xb)
                pred = slogits.argmax(1)
                tot += yb.size(0); cor += (pred == yb).sum().item()
        v_acc = 100.0 * cor / max(1, tot)

        print(f"ep {ep:02d} | train_loss={loss_sum/len(tr_dl):.3f} | val_acc={v_acc:.2f}%")
        if v_acc > best_val:
            best_val = v_acc
            best_state = {k: v.detach().cpu().clone() for k,v in student.state_dict().items()}
            stall = 0
        else:
            stall += 1
            if stall >= distillation_patience:
                print(f"Early stopping at ep {ep} (best {best_val:.2f}%)")
                break
        log_logit_std(student, tr_dl, device, prefix="train", max_batches=2)
        log_logit_std(student, va_dl, device, prefix="val",   max_batches=2)
        # val per class
        cls_acc = eval_per_class(student, va_dl, num_classes, device)
        print("\nPer-class (student on cached features):")
        for i, acc in enumerate(cls_acc):
            print(f"  class {i:2d}: {acc:5.1f}%")
        print(f"Overall val acc (student): {best_val:.2f}%")

    return best_state, best_val

################### MAIN FUNCTION!!!!!!
##################
def distillation_1():
    # -------------- config --------------
    cfg  = {
    # Distillation part 1
    "cache_dir": "/content/backbone_artifacts/teacher_cache",   # where train_cache.pt / val_cache.pt live
    "out_dir": "/content/distillation_1/",
    "feat_dim_student": 128,  # must match CIL feat_dim

    # Training settings
    "distillation_batch_size_train": 256,
    "distillation_batch_size_val": 256,
    "distillation_epochs": 50,
    "distillation_patience": 8,
    "distillation_lr": 5e-4,
    "distillation_weight_decay": 1e-4,
    "distillation_dropout": 0.3,

    # Loss weights
    "distillation_label_smoothing": 0.05,
    "distillation_alpha_kd": 1.0,   # KL distillation (teacher logits)
    "distillation_kd_T": 2.0,       # temperature
    "distillation_alpha_ce": 0.0,   # CE loss weight
    "distillation_alpha_feat": 0.0, # feature MSE weight
}
    cache_dir = cfg['cache_dir']
    out_dir = cfg['out_dir']
    feat_dim_student = cfg['feat_dim_student']
    distillation_batch_size_train = cfg['distillation_batch_size_train']
    distillation_batch_size_val = cfg['distillation_batch_size_val']
    distillation_epochs = cfg['distillation_epochs']
    distillation_patience = cfg['distillation_patience']
    distillation_lr = cfg['distillation_lr']
    distillation_weight_decay = cfg['distillation_weight_decay']
    distillation_label_smoothing = cfg['distillation_label_smoothing']
    distillation_alpha_kd = cfg['distillation_alpha_kd']
    distillation_kd_T = cfg['distillation_kd_T']
    distillation_alpha_ce = cfg['distillation_alpha_ce']
    distillation_alpha_feat = cfg['distillation_alpha_feat']
    distillation_dropout = 0.3

    # NOTE: if distillation_alpha_feat and distillation_alpha_ce > 0 , we are no longer using class "logit only distillation"
    # we have
    # - KD term (teacher soft labels) controlled by distillation_alpha_kd, KD_T
	# - CE term (ground truth) controlled by distillation_alpha_ce
	# - Feature term (rep alignment) controlled by distillation_alpha_feat

    # alpha_kd = how much I care about KD?
	# kd_T = how soft is the teacher’s signal in KD?

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # -------------- load caches --------------
    cache_dir = Path(cache_dir)
    train_cache = torch.load(cache_dir/"train_cache.pt", map_location="cpu")
    val_cache   = torch.load(cache_dir/"val_cache.pt",   map_location="cpu")


    # ----------------load tensors --------------
    xtr = train_cache["features"].float()    # [Ntr, 1280]  teacher penult features
    ytr = train_cache["y_local"].long()     # [Ntr]
    tlog_tr = train_cache["logits"].float()  # [Ntr, B]

    xva = val_cache["features"].float()      # [Nva, 1280]
    yva = val_cache["y_local"].long()       # [Nva]
    tlog_va = val_cache["logits"].float()    # [Nva, B]

    meta = train_cache["meta"]
    local_ids = meta["backbone_gids"]               # global IDs order used in head
    num_classes = tlog_tr.shape[1]                  # B
    feat_teacher = xtr.shape[1]                     # 1280

    print(f"Train: feats={tuple(xtr.shape)} logits={tuple(tlog_tr.shape)}")
    print(f" Val : feats={tuple(xva.shape)} logits={tuple(tlog_va.shape)}")
    print(f"Classes: {num_classes} | teacher feat dim: {feat_teacher}")


    print("[sanity] tlog mean/std:", tlog_tr.mean().item(), tlog_tr.std().item())



    # -------------- student on cached features --------------
    # Learn a small projector (1280→feat_dim) + KD head.
    # After training, we save only the projector weights as “proj” (for tiny CNN it’s the same shape).

    student = StudentOnCache(in_dim=feat_teacher, feat_dim=feat_dim_student,
                         num_classes=num_classes, drop= distillation_dropout).to(device)

    # -------------- loaders --------------
    tr_ds = TensorDataset(xtr, ytr, tlog_tr)
    va_ds = TensorDataset(xva, yva, tlog_va)
    tr_dl = DataLoader(tr_ds, batch_size=distillation_batch_size_train, shuffle=True)
    va_dl = DataLoader(va_ds, batch_size=distillation_batch_size_val, shuffle=False)

    # -------------- losses --------------

    ce = nn.CrossEntropyLoss(label_smoothing=distillation_label_smoothing)
    mse = nn.MSELoss()

    opt = torch.optim.AdamW(student.parameters(), lr=distillation_lr, weight_decay=distillation_weight_decay)

    best_state, best_val = train(student, opt,
                       distillation_epochs,
                       device, tr_dl, va_dl, distillation_alpha_kd,
                       distillation_alpha_ce, distillation_kd_T, ce, distillation_alpha_feat,
                       mse, distillation_patience, num_classes)

    if best_state is not None:
        student.load_state_dict(best_state, strict=True)




    # -------------- save artifacts for CIL --------------
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)

    # Only need projector weights; the CIL tiny CNN backbone outputs 128 then proj→feat_dim in CIL.
    save = {
        "proj_from_teacher_penult": {k.replace("proj.", ""): v for k, v in student.state_dict().items() if k.startswith("proj.")},
        "kd_head": {k.replace("kd_head.", ""): v for k, v in student.state_dict().items() if k.startswith("kd_head.")},
        "meta": {
            "teacher_penult_dim": feat_teacher,
            "student_feat_dim": feat_dim_student,
            "backbone_gids": local_ids,
            "val_best": float(best_val),
        }
    }
    torch.save(save, out/"student_from_cache.pth")
    (out/"student_from_cache_meta.json").write_text(json.dumps(save["meta"], indent=2))

    print("\nSaved:")
    print(f"  • {out/'student_from_cache.pth'}")
    print(f"  • {out/'student_from_cache_meta.json'}")


distillation_1()


# DISTILLATION PART 2

In [ ]:
 # distillation_mcu_friendly_tuned.py
# Tuned version of MCU-friendly distillation with better hyperparameters
# Based on distillation_mcu_friendly.py but with improvements

import json, random
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from pathlib import Path
from tqdm import tqdm

### UTILS

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def canon_key_from_path(p: str) -> str:
    p = Path(p)
    parts = [*p.parts]
    if "Images" in parts:
        i = parts.index("Images")
        rel = parts[i+1:]
    else:
        rel = parts[-2:]
    return "/".join(rel).replace("\\", "/")

def build_cache_maps(cache: dict):
    keys = [k.replace("\\","/") for k in cache["keys"]]
    logits = cache["logits"]
    y = cache["y_local"]
    fts = cache["feat_targets"]
    map_logits = {k: logits[i] for i,k in enumerate(keys)}
    map_y = {k: int(y[i]) for i,k in enumerate(keys)}
    map_ft = {k: fts[i] for i,k in enumerate(keys)}
    return map_ft, map_logits, map_y

@torch.no_grad()
def add_feat_targets(cache, P):
    feats = cache["features"].float()
    cache["feat_targets"] = P(feats).float()
    return cache

### IMPROVED MCU STUDENT ARCHITECTURE
def CBR3(cin, cout, stride=1):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, stride=stride, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )

def CBR1(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )

class TunedMCUStudentCNN(nn.Module):
    """
    M3: Old backbone, cheaper compute
    - 1st conv uses stride=2 (drop first MaxPool)
    - Second 3x3 in each block replaced with 1x1
    - Small width bump in the last stage (to 160) to recover accuracy
    """
    def __init__(self, in_channels=3, feat_dim=128, num_classes=10, img_size=160):
        super().__init__()

        # Create backbone as a single Sequential (this is what Colab code expects)
        self.backbone = nn.Sequential(
            # Stage 1: 160 -> 80 (stride=2)  [no pool here]
            CBR3(in_channels, 32, stride=2),  # downsample early
            CBR1(32, 32),                     # 1x1 instead of second 3x3
            
            # Stage 2: 80 -> 40
            CBR3(32, 64, stride=1),
            CBR1(64, 64),
            nn.MaxPool2d(2, 2),
            
            # Stage 3: 40 -> 20
            CBR3(64, 128, stride=1),
            CBR1(128, 128),
            nn.MaxPool2d(2, 2),
            
            # Stage 4: 20 -> 10  (slight width bump to help accuracy)
            CBR3(128, 160, stride=1),
            CBR1(160, 160),
            nn.MaxPool2d(2, 2),
        )

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(nn.Dropout(0.20), nn.Linear(160, feat_dim))
        self.classifier = nn.Linear(feat_dim, num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if getattr(m, 'bias', None) is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def _features(self, x):
        x = self.backbone(x)
        x = self.gap(x).flatten(1)  # (N, 160)
        return x

    def forward(self, x):
      feats = self._features(x)
      z = self.proj(feats)             # (N, feat_dim)
      logits = self.classifier(z)      # (N, num_classes)
      return logits, z  # Return both logits and projected features for distillation


### IMPROVED LOSS FUNCTIONS

def tuned_kd_loss(s_logits, t_logits, T=2.5, alpha=0.8):
    """Improved KD loss with better temperature and alpha balance"""
    # Soft targets
    soft_loss = F.kl_div(
        F.log_softmax(s_logits/T, dim=1),
        F.softmax(t_logits/T, dim=1),
        reduction="batchmean"
    ) * (T * T)

    # Hard targets (ground truth)
    hard_loss = F.cross_entropy(s_logits, t_logits.argmax(dim=1))

    return alpha * soft_loss + (1 - alpha) * hard_loss

def tuned_feature_loss(s_feat, t_feat):
    """Improved feature matching loss"""
    s_norm = F.normalize(s_feat, dim=1)
    t_norm = F.normalize(t_feat, dim=1)
    return 1.0 - F.cosine_similarity(s_norm, t_norm, dim=1).mean()

### DATASET

class DogWithCacheTargets(Dataset):
    def __init__(self, df, split, tfm, gid_to_local, map_logits, map_y, map_ft):
        self.df = df[df["split"] == split].reset_index(drop=True)
        self.tfm = tfm
        self.g2l = gid_to_local
        self.map_logits = map_logits
        self.map_y = map_y
        self.map_ft = map_ft

        self.keys = [canon_key_from_path(r.img_path) for _, r in self.df.iterrows()]
        missing = [k for k in self.keys if k not in self.map_logits]
        if missing:
            raise RuntimeError(f"{len(missing)} images missing in cache for split={split}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(r.img_path).convert("RGB")
        x = self.tfm(img)
        y = self.g2l[int(r.gid)]
        key = self.keys[i]
        tlog = self.map_logits[key]
        tft = self.map_ft[key]
        return x, y, tlog, tft

### MAIN DISTILLATION FUNCTION

def tuned_mcu_distillation():
    """Tuned MCU-friendly distillation with improved hyperparameters"""

    # Improved configuration for better performance
    CFG = {
        "index_csv": "/content/dog_breeds_index.csv",
        "teacher_meta": "/content/backbone_artifacts/backbone_meta.json",
        "cache_dir": "/content/backbone_artifacts/teacher_cache",
        "train_cache": "train_cache.pt",
        "val_cache": "val_cache.pt",
        "proj_ckpt": "/content/distillation_1/student_from_cache.pth",
        "out_dir": "/content/distillation_mcu_tuned/",

        # MCU-friendly student model (same size)
        "feat_dim": 128,
        "num_classes": 10,
        "img_size": 160,

        # Improved training parameters
        "seed": 42,
        "batch_size_train": 32,  # Larger batch for better gradients
        "batch_size_val": 32,
        "epochs": 120,  # More epochs
        "patience": 25,  # More patience
        "lr": 5e-4,  # Lower learning rate for stability
        "weight_decay": 5e-5,  # Reduced weight decay

        # Improved distillation parameters
        "kd_T": 2.5,  # Better temperature
        "kd_alpha": 0.8,  # Better alpha balance
        "kd_weight": 1.0,
        "feat_weight": 0.5,  # Increased feature weight
        "ce_weight": 0.3,  # Increased CE weight

        "num_workers": 2,
    }

    set_seed(CFG["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("=== Tuned MCU-Friendly Knowledge Distillation ===")
    print(f"Student: {CFG['feat_dim']}D features, {CFG['num_classes']} classes, {CFG['img_size']}x{CFG['img_size']} images")
    print("Improved hyperparameters for better performance")

    # Load meta and caches
    meta = json.loads(Path(CFG["teacher_meta"]).read_text())
    mean = meta["normalization"]["mean"]
    std = meta["normalization"]["std"]
    backbone_gids = list(map(int, meta["backbone_gids"]))

    cache_dir = Path(CFG["cache_dir"])
    train_cache = torch.load(cache_dir/CFG["train_cache"], map_location="cpu")
    val_cache = torch.load(cache_dir/CFG["val_cache"], map_location="cpu")
    out_dir = Path(CFG["out_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)

    train_cache["keys"] = [canon_key_from_path(k) for k in train_cache["keys"]]
    val_cache["keys"] = [canon_key_from_path(k) for k in val_cache["keys"]]

    print(f"Train cache: {len(train_cache['keys'])} samples")
    print(f"Val cache: {len(val_cache['keys'])} samples")

    # Build feature targets using projector
    saved = torch.load(CFG["proj_ckpt"], map_location="cpu")
    proj_sd = saved["proj_from_teacher_penult"]

    # Create a new projector that maps to student feature dimension
    P = nn.Linear(train_cache["features"].shape[1], CFG["feat_dim"])
    P.weight.data.copy_(proj_sd["1.weight"][:CFG["feat_dim"], :])
    P.bias.data.copy_(proj_sd["1.bias"][:CFG["feat_dim"]])
    P = P.cpu().eval()

    train_cache = add_feat_targets(train_cache, P)
    val_cache = add_feat_targets(val_cache, P)

    # Build datasets
    tr_map_ft, tr_map_logits, tr_map_y = build_cache_maps(train_cache)
    va_map_ft, va_map_logits, va_map_y = build_cache_maps(val_cache)

    df = pd.read_csv(CFG["index_csv"])
    df = df[df["gid"].isin(backbone_gids)].copy()
    gid_to_local = {g:i for i,g in enumerate(backbone_gids)}

    # Simple transforms for on-device training
    train_tf = transforms.Compose([
        transforms.Resize((CFG["img_size"], CFG["img_size"])),
        transforms.RandomHorizontalFlip(p=0.5),  # Only basic flip
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    val_tf = transforms.Compose([
        transforms.Resize((CFG["img_size"], CFG["img_size"])),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    ds_tr = DogWithCacheTargets(df, "train", train_tf, gid_to_local, tr_map_logits, tr_map_y, tr_map_ft)
    ds_va = DogWithCacheTargets(df, "test", val_tf, gid_to_local, va_map_logits, va_map_y, va_map_ft)

    dl_tr = DataLoader(ds_tr, batch_size=CFG["batch_size_train"], shuffle=True, num_workers=CFG["num_workers"], pin_memory=pin)
    dl_va = DataLoader(ds_va, batch_size=CFG["batch_size_val"], shuffle=False, num_workers=CFG["num_workers"], pin_memory=pin)

    # Build tuned MCU student model
    student = TunedMCUStudentCNN(
        in_channels=3,
        feat_dim=CFG["feat_dim"],
        num_classes=CFG["num_classes"],
        img_size=CFG["img_size"]
    ).to(device)

    # Count parameters
    total_params = sum(p.numel() for p in student.parameters())
    print(f"Student parameters: {total_params:,}")
    print(f"Estimated model size: ~{total_params * 4 / 1024:.1f}KB (FP32)")
    print(f"After quantization (INT8): ~{total_params / 1024:.1f}KB")
    print(f"After 50% pruning + INT8: ~{total_params / 2 / 1024:.1f}KB")

    # Improved optimizer and scheduler
    optimizer = torch.optim.AdamW(
        student.parameters(),
        lr=CFG["lr"],
        weight_decay=CFG["weight_decay"]
    )

    # Better scheduler with warmup
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=CFG["lr"],
        epochs=CFG["epochs"],
        steps_per_epoch=len(dl_tr),
        pct_start=0.1,  # 10% warmup
        anneal_strategy='cos'
    )

    # Training loop
    best_val_acc = 0.0
    best_state = None
    patience_counter = 0

    print("\nStarting tuned MCU distillation training...")

    for epoch in range(CFG["epochs"]):
        # Training phase
        student.train()
        train_loss = 0.0
        train_kd_loss = 0.0
        train_feat_loss = 0.0
        train_ce_loss = 0.0
        batch_iter = tqdm(enumerate(dl_tr), total = len(dl_tr), desc=f"On epoch: {epoch+1}/{CFG['epochs']}")
        for batch_idx, (xb, yb, tlog_cpu, tfeat_cpu) in batch_iter:
            xb = xb.to(device)
            yb = yb.to(device)
            tlog = tlog_cpu.to(device)
            tfeat = tfeat_cpu.to(device)

            # Forward pass
            slogits, sfeat = student(xb)

            # Compute losses
            kd_loss = tuned_kd_loss(slogits, tlog, CFG["kd_T"], CFG["kd_alpha"])
            feat_loss = tuned_feature_loss(sfeat, tfeat)
            ce_loss = F.cross_entropy(slogits, yb)

            # Combined loss
            total_loss = (CFG["kd_weight"] * kd_loss +
                         CFG["feat_weight"] * feat_loss +
                         CFG["ce_weight"] * ce_loss)

            # Backward pass
            optimizer.zero_grad()
            total_loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()

            # Log losses
            train_loss += total_loss.item()
            train_kd_loss += kd_loss.item()
            train_feat_loss += feat_loss.item()
            train_ce_loss += ce_loss.item()

        # Validation phase
        student.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0.0

        with torch.no_grad():
            for xb, yb, tlog_cpu, tfeat_cpu in dl_va:
                xb = xb.to(device)
                yb = yb.to(device)
                tlog = tlog_cpu.to(device)
                tfeat = tfeat_cpu.to(device)

                slogits, sfeat = student(xb)

                # Compute validation loss
                kd_loss = tuned_kd_loss(slogits, tlog, CFG["kd_T"], CFG["kd_alpha"])
                feat_loss = tuned_feature_loss(sfeat, tfeat)
                ce_loss = F.cross_entropy(slogits, yb)

                val_loss += (CFG["kd_weight"] * kd_loss +
                           CFG["feat_weight"] * feat_loss +
                           CFG["ce_weight"] * ce_loss).item()

                # Compute accuracy
                _, predicted = slogits.max(1)
                val_total += yb.size(0)
                val_correct += predicted.eq(yb).sum().item()

        val_acc = 100.0 * val_correct / val_total
        avg_train_loss = train_loss / len(dl_tr)
        avg_val_loss = val_loss / len(dl_va)

        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}
            patience_counter = 0

            #  CHECKPOINTING EVERY EPOCH!!! --> Load best model
            if best_state is not None:
                student.load_state_dict(best_state)

            print(f"\nBest validation accuracy: {best_val_acc:.2f}%")

            # Save the tuned student

            save_dict = {
                "backbone": {k.replace("backbone.", ""): v for k, v in student.state_dict().items() if k.startswith("backbone.")},
                "proj": {k.replace("proj.", ""): v for k, v in student.state_dict().items() if k.startswith("proj.")},
                "classifier": {k.replace("classifier.", ""): v for k, v in student.state_dict().items() if k.startswith("classifier.")},
            }

            torch.save(save_dict, out_dir / "tuned_mcu_student.pth")
        else:
            patience_counter += 1

        # Print progress
        if epoch % 10 == 0 or epoch == CFG["epochs"] - 1:
            print(f"Epoch {epoch:3d}/{CFG['epochs']} | "
                  f"Train Loss: {avg_train_loss:.4f} | "
                  f"Val Loss: {avg_val_loss:.4f} | "
                  f"Val Acc: {val_acc:.2f}% | "
                  f"Best: {best_val_acc:.2f}% | "
                  f"LR: {optimizer.param_groups[0]['lr']:.6f}")

        if patience_counter >= CFG["patience"]:
            print(f"Early stopping at epoch {epoch}")
            break



    meta_out = {
        "student_arch": "tuned_mcu_cnn",
        "feature_dim": CFG["feat_dim"],
        "img_size": CFG["img_size"],
        "normalization": {"mean": mean, "std": std},
        "backbone_gids": backbone_gids,
        "best_val_acc": float(best_val_acc),
        "total_params": total_params,
        "model_size_fp32_kb": total_params * 4 / 1024,
        "model_size_int8_kb": total_params / 1024,
        "model_size_pruned_int8_kb": total_params / 2 / 1024,  # 50% pruning estimate
    }

    (out_dir / "tuned_mcu_student_meta.json").write_text(json.dumps(meta_out, indent=2))

    print(f"\nSaved tuned MCU student to: {out_dir}")
    print(f"Model size: {total_params:,} parameters")
    print(f"FP32 size: ~{total_params * 4 / 1024:.1f}KB")
    print(f"INT8 size: ~{total_params / 1024:.1f}KB")
    print(f"After 50% pruning + INT8: ~{total_params / 2 / 1024:.1f}KB")
    print("Ready for pruning and quantization!")
    print("Done!")

if __name__ == "__main__":
    tuned_mcu_distillation()
